# Bias-Variance Trade-off & Model Diagnostics (No User-Defined Functions)

Same coverage as before, but every step is written **directly**, using scikit-learn's
built-in implementations — no `def` anywhere:

1. Bias vs. Variance intuition (underfitting vs. overfitting vs. good fit)
2. Train vs. validation error and the **generalization gap**
3. The **Bias-Variance Diagnostic Matrix**
4. **Learning curves** (`sklearn.model_selection.learning_curve`)
5. **Validation curves** (`sklearn.model_selection.validation_curve`)
6. Cross-validation stability (`sklearn.model_selection.cross_val_score`)


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, learning_curve, validation_curve, cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

np.random.seed(42)
plt.rcParams["figure.figsize"] = (10, 4.5)


## 2. Synthetic Dataset

A nonlinear ground-truth function plus noise — a good sandbox for showing how model
**complexity** trades off bias against variance.


In [ ]:
n_samples = 60
X = np.sort(np.random.rand(n_samples))
y = np.sin(1.5 * np.pi * X) + np.random.normal(0, 0.25, n_samples)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=42)

X_train_2d = X_train.reshape(-1, 1)
X_val_2d = X_val.reshape(-1, 1)
X_all_2d = X.reshape(-1, 1)
x_grid = np.linspace(0, 1, 200).reshape(-1, 1)

plt.scatter(X_train, y_train, label="Train", alpha=0.7)
plt.scatter(X_val, y_val, label="Validation", alpha=0.7)
plt.plot(x_grid, np.sin(1.5 * np.pi * x_grid), "k--", label="True function")
plt.legend()
plt.title("Synthetic Nonlinear Dataset")
plt.tight_layout()
plt.show()


## 3. Underfitting vs. Overfitting vs. Good Fit

We control model complexity with the **degree of a polynomial** fit to the data:
- Degree 1 (a straight line) is too simple → **underfitting / high bias**
- Degree 15 is far too flexible → **overfitting / high variance**
- Degree 4 is about right → **good fit**

Each model is built directly with `make_pipeline(PolynomialFeatures, LinearRegression)` — no wrapper function.


In [ ]:
underfit_model = make_pipeline(PolynomialFeatures(degree=1), LinearRegression())
underfit_model.fit(X_train_2d, y_train)

goodfit_model = make_pipeline(PolynomialFeatures(degree=4), LinearRegression())
goodfit_model.fit(X_train_2d, y_train)

overfit_model = make_pipeline(PolynomialFeatures(degree=15), LinearRegression())
overfit_model.fit(X_train_2d, y_train)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)

axes[0].scatter(X_train, y_train, alpha=0.6, label="Train")
axes[0].scatter(X_val, y_val, alpha=0.6, label="Validation")
axes[0].plot(x_grid, underfit_model.predict(x_grid), color="red", label="Model")
axes[0].plot(x_grid, np.sin(1.5 * np.pi * x_grid), "k--", alpha=0.5, label="True function")
axes[0].set_title("Underfit (degree 1)")
axes[0].set_ylim(-2, 2)
axes[0].legend(loc="lower left", fontsize=8)

axes[1].scatter(X_train, y_train, alpha=0.6)
axes[1].scatter(X_val, y_val, alpha=0.6)
axes[1].plot(x_grid, goodfit_model.predict(x_grid), color="red")
axes[1].plot(x_grid, np.sin(1.5 * np.pi * x_grid), "k--", alpha=0.5)
axes[1].set_title("Good Fit (degree 4)")

axes[2].scatter(X_train, y_train, alpha=0.6)
axes[2].scatter(X_val, y_val, alpha=0.6)
axes[2].plot(x_grid, overfit_model.predict(x_grid), color="red")
axes[2].plot(x_grid, np.sin(1.5 * np.pi * x_grid), "k--", alpha=0.5)
axes[2].set_title("Overfit (degree 15)")

plt.tight_layout()
plt.show()


### Train vs. Validation Error for Each Model

In [ ]:
underfit_train_mse = mean_squared_error(y_train, underfit_model.predict(X_train_2d))
underfit_val_mse = mean_squared_error(y_val, underfit_model.predict(X_val_2d))

goodfit_train_mse = mean_squared_error(y_train, goodfit_model.predict(X_train_2d))
goodfit_val_mse = mean_squared_error(y_val, goodfit_model.predict(X_val_2d))

overfit_train_mse = mean_squared_error(y_train, overfit_model.predict(X_train_2d))
overfit_val_mse = mean_squared_error(y_val, overfit_model.predict(X_val_2d))

diagnostics_df = pd.DataFrame({
    "Model": ["Underfit (degree 1)", "Good Fit (degree 4)", "Overfit (degree 15)"],
    "Train MSE": [underfit_train_mse, goodfit_train_mse, overfit_train_mse],
    "Validation MSE": [underfit_val_mse, goodfit_val_mse, overfit_val_mse],
})
diagnostics_df["Gap (Val - Train)"] = diagnostics_df["Validation MSE"] - diagnostics_df["Train MSE"]
diagnostics_df


## 4. Bias-Variance Diagnostic Matrix

Classified directly with `pandas`/`numpy` conditions (`np.select`) instead of a
hand-written diagnosis function.


In [ ]:
high_error_threshold = 0.15

conditions = [
    (diagnostics_df["Train MSE"] > high_error_threshold) & (diagnostics_df["Validation MSE"] > high_error_threshold),
    (diagnostics_df["Train MSE"] <= high_error_threshold) & (diagnostics_df["Validation MSE"] > high_error_threshold),
    (diagnostics_df["Train MSE"] < 0.02) & (diagnostics_df["Validation MSE"] < 0.02),
]
choices = ["High bias / Underfitting", "High variance / Overfitting", "Good fit or investigate leakage"]

diagnostics_df["Diagnosis"] = np.select(conditions, choices, default="Good fit")
diagnostics_df


## 5. Learning Curves

Does getting **more training data** help? `sklearn.model_selection.learning_curve` is
called directly for each model — no wrapper function.


In [ ]:
train_sizes_range = np.linspace(0.2, 1.0, 8)
cv_splitter = KFold(n_splits=5, shuffle=True, random_state=42)

underfit_sizes, underfit_train_scores, underfit_val_scores = learning_curve(
    make_pipeline(PolynomialFeatures(1), LinearRegression()), X_all_2d, y,
    train_sizes=train_sizes_range, cv=cv_splitter, scoring="neg_mean_squared_error",
)

goodfit_sizes, goodfit_train_scores, goodfit_val_scores = learning_curve(
    make_pipeline(PolynomialFeatures(4), LinearRegression()), X_all_2d, y,
    train_sizes=train_sizes_range, cv=cv_splitter, scoring="neg_mean_squared_error",
)

overfit_sizes, overfit_train_scores, overfit_val_scores = learning_curve(
    make_pipeline(PolynomialFeatures(15), LinearRegression()), X_all_2d, y,
    train_sizes=train_sizes_range, cv=cv_splitter, scoring="neg_mean_squared_error",
)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].plot(underfit_sizes, -underfit_train_scores.mean(axis=1), "o-", label="Training error")
axes[0].plot(underfit_sizes, -underfit_val_scores.mean(axis=1), "o-", label="Validation error")
axes[0].set_title("Underfit (degree 1)")
axes[0].set_xlabel("Training set size")
axes[0].set_ylabel("MSE")
axes[0].legend()

axes[1].plot(goodfit_sizes, -goodfit_train_scores.mean(axis=1), "o-", label="Training error")
axes[1].plot(goodfit_sizes, -goodfit_val_scores.mean(axis=1), "o-", label="Validation error")
axes[1].set_title("Good Fit (degree 4)")
axes[1].set_xlabel("Training set size")

axes[2].plot(overfit_sizes, -overfit_train_scores.mean(axis=1), "o-", label="Training error")
axes[2].plot(overfit_sizes, -overfit_val_scores.mean(axis=1), "o-", label="Validation error")
axes[2].set_title("Overfit (degree 15)")
axes[2].set_xlabel("Training set size")

plt.tight_layout()
plt.show()


**Reading the curves:**
- **Underfit:** both curves are high and close together — more data won't help much; the model itself is too simple.
- **Overfit:** a persistent gap between low training error and high validation error — more (representative) data can help narrow this gap.
- **Good fit:** the curves converge to a low, similar error as training size grows.


## 6. Validation Curves

Does the **hyperparameter value** need to change? `sklearn.model_selection.validation_curve`
sweeps `max_depth` of a `DecisionTreeRegressor` directly.


In [ ]:
max_depth_range = [1, 2, 3, 4, 5, 6, 8, 10, 15, 20]

vc_train_scores, vc_val_scores = validation_curve(
    DecisionTreeRegressor(random_state=42),
    X_all_2d, y,
    param_name="max_depth",
    param_range=max_depth_range,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring="neg_mean_squared_error",
)

vc_train_mse = -vc_train_scores.mean(axis=1)
vc_val_mse = -vc_val_scores.mean(axis=1)

plt.plot(max_depth_range, vc_train_mse, "o-", label="Training error")
plt.plot(max_depth_range, vc_val_mse, "o-", label="Validation error")
plt.xlabel("max_depth")
plt.ylabel("MSE")
plt.title("Validation Curve: Decision Tree max_depth")
plt.legend()
plt.tight_layout()
plt.show()

best_depth = max_depth_range[np.argmin(vc_val_mse)]
print("Best max_depth by validation error:", best_depth)


**Reading the curve:**
- Low `max_depth` → both errors high → **underfitting**.
- High `max_depth` → training error keeps falling but validation error rises → **overfitting**.
- The dip in the validation curve marks the sweet spot.


## 7. Learning Curve vs. Validation Curve — Summary Table

In [ ]:
comparison_table = pd.DataFrame({
    "Property": ["X-axis", "Main Question", "Detects Underfitting", "Detects Overfitting", "Tunes Hyperparameter", "Example"],
    "Learning Curve": ["Training size", "Do I need more data?", "Yes", "Yes", "No", "100 -> 1000 samples"],
    "Validation Curve": ["Hyperparameter value", "Is the value appropriate?", "Yes", "Yes", "Helps diagnose/select range", "max_depth: 1-20"],
})
comparison_table


## 8. Cross-Validation Stability (Mean vs. Std)

A model's average cross-validation score isn't the whole story — the **spread** across
folds tells you how stable that performance is. `cross_val_score` is called directly
for each model.


In [ ]:
cv_splitter = KFold(n_splits=5, shuffle=True, random_state=42)

goodfit_cv_scores = cross_val_score(
    make_pipeline(PolynomialFeatures(4), LinearRegression()), X_all_2d, y, cv=cv_splitter, scoring="r2"
)
overfit_cv_scores = cross_val_score(
    make_pipeline(PolynomialFeatures(15), LinearRegression()), X_all_2d, y, cv=cv_splitter, scoring="r2"
)

print("Good-fit model (degree 4)  fold scores:", np.round(goodfit_cv_scores, 3))
print("Good-fit model (degree 4)  mean R2 =", goodfit_cv_scores.mean(), " std =", goodfit_cv_scores.std())
print()
print("Overfit model (degree 15)  fold scores:", np.round(overfit_cv_scores, 3))
print("Overfit model (degree 15)  mean R2 =", overfit_cv_scores.mean(), " std =", overfit_cv_scores.std())


In [ ]:
stability_df = pd.DataFrame({
    "Model": ["Good Fit (degree 4)", "Overfit (degree 15)"],
    "Mean R2": [goodfit_cv_scores.mean(), overfit_cv_scores.mean()],
    "Std R2": [goodfit_cv_scores.std(), overfit_cv_scores.std()],
})
stability_df["Stability"] = np.where(stability_df["Std R2"] < 0.15, "Stable across folds", "Unstable — investigate variance")
stability_df


**Note:** a high standard deviation across folds isn't automatically the same thing
as theoretical high variance, but it's an important stability warning — it means
performance depends heavily on which data ended up in which fold.


## 9. Key Takeaways

- **High bias (underfitting):** training and validation error both high, small gap.
- **High variance (overfitting):** training error low, validation error high, large gap.
- **Good fit:** both errors low and close together.
- **Extremely low** training and validation error together can mean a great fit — or a **data leakage** problem worth investigating.
- **Learning curves** answer "would more data help?" — most useful for overfitting.
- **Validation curves** answer "is this hyperparameter value appropriate?" — useful for both underfitting and overfitting.
- Always check **std alongside mean** in cross-validation: low std means stable performance; high std is a stability warning even if the mean score looks fine.
